In [ ]:
# -*- coding: utf-8 -*-
"""
BrainVital8 Calibration Analysis
==================================
Calibration plots, forest plots, and summary table across 6 cohorts.

Cohort time points:
  UKB: 10y, CHARLS: 5y, ELSA: 7y, HRS: 7y, SHARE: 10y, KLoSA: 5y

Outputs:
  1. Figure 1: 2x3 calibration plots
  2. Figure 2: Forest plot (4 panels: Slope, O:E, C-stat, IPA)
  3. Individual large plots (supplementary)
  4. Table 1: Summary CSV
"""

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.unicode_minus": False,
    "axes.linewidth": 1.2,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
})


# ========================= Configuration =========================
DATA_FILE = r"./data/SEM_Analysis/All_cohort_final_BrainVital8.csv"
OUTPUT_DIR = r"./results/SEM_Analysis/publication_figures"

COHORT_CONFIG = {
    "ukb":    {"time_point": 10, "color": "#e74c3c", "label": "UKB"},
    "charls": {"time_point": 5,  "color": "#3498db", "label": "CHARLS"},
    "elsa":   {"time_point": 7,  "color": "#2ecc71", "label": "ELSA"},
    "hrs":    {"time_point": 7,  "color": "#f39c12", "label": "HRS"},
    "share":  {"time_point": 10, "color": "#9b59b6", "label": "SHARE"},
    "klosa":  {"time_point": 5,  "color": "#c0392b", "label": "KLoSA"},
}

COVARIATES = ["baseline_age", "gender", "bmi", "drinking", "hypertension", "scores"]
N_FOLDS = 5
N_GROUPS = 10
RANDOM_SEED = 42


# ========================= Data Loading =========================

def load_and_split(filepath):
    """Load data, rename columns, split by cohort."""
    df = pd.read_csv(filepath)
    df = df.rename(columns={
        "age": "baseline_age", "sex": "gender",
        "drink": "drinking", "BrainVital8": "scores",
    })
    for col in COVARIATES + ["time", "status"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df[df["time"] > 0]
    df["cohort"] = df["cohort"].str.lower().str.strip()

    cohorts = {}
    for name in sorted(df["cohort"].unique()):
        sub = df[df["cohort"] == name][COVARIATES + ["time", "status"]].dropna().reset_index(drop=True)
        cohorts[name] = sub
    return cohorts


# ========================= Prediction =========================

def get_oof_predictions(df, covariates, target_year, k=5):
    """Get out-of-fold risk predictions using Cox regression with K-fold CV."""
    pred = np.zeros(len(df))
    kf = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)
    for tr_idx, va_idx in kf.split(df):
        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_va = df.iloc[va_idx].reset_index(drop=True)
        try:
            cph = CoxPHFitter()
            cph.fit(df_tr[covariates + ["time", "status"]],
                    duration_col="time", event_col="status")
            surv = cph.predict_survival_function(df_va, times=[target_year])
            pred[va_idx] = 1 - surv.values.flatten()
        except Exception:
            pred[va_idx] = df["status"].mean()
    return np.clip(pred, 0, 1)


# ========================= Calibration =========================

def km_based_calibration(pred_risk, time, status, target_year, n_groups=10):
    """Compute KM-based calibration by risk decile groups.
    Returns dict with slope, intercept, O:E ratio, and calibration data.
    """
    df = pd.DataFrame({"pred": pred_risk, "time": time, "status": status})
    try:
        df["group"] = pd.qcut(df["pred"], q=n_groups, labels=False, duplicates="drop")
    except ValueError:
        df["group"] = pd.qcut(df["pred"], q=5, labels=False, duplicates="drop")

    results = []
    for g in sorted(df["group"].unique()):
        sub = df[df["group"] == g]
        if len(sub) < 5:
            continue
        kmf = KaplanMeierFitter()
        try:
            kmf.fit(sub["time"], sub["status"])
            if target_year > sub["time"].max():
                continue
            try:
                surv_t = kmf.survival_function_at_times(target_year).values[0]
            except Exception:
                idx = np.argmin(np.abs(kmf.survival_function_.index - target_year))
                surv_t = kmf.survival_function_.iloc[idx, 0]
            try:
                ci_df = kmf.confidence_interval_survival_function_
                idx = np.argmin(np.abs(ci_df.index - target_year))
                obs_lower = 1 - ci_df.iloc[idx, 1]
                obs_upper = 1 - ci_df.iloc[idx, 0]
            except Exception:
                obs_lower = obs_upper = np.nan
            results.append({
                "n": len(sub), "predicted": sub["pred"].mean(),
                "observed": 1 - surv_t, "obs_lower": obs_lower, "obs_upper": obs_upper,
            })
        except Exception:
            continue

    if len(results) < 3:
        return None
    cal_df = pd.DataFrame(results)
    cal_df = cal_df[(cal_df["observed"] > 0) & (cal_df["observed"] < 1)]
    if len(cal_df) < 3:
        return None

    # Weighted linear regression: observed ~ predicted
    reg = LinearRegression()
    reg.fit(cal_df["predicted"].values.reshape(-1, 1),
            cal_df["observed"].values, sample_weight=cal_df["n"].values)

    # Overall O:E ratio
    kmf_all = KaplanMeierFitter()
    kmf_all.fit(time, status)
    try:
        surv_all = kmf_all.survival_function_at_times(target_year).values[0]
    except Exception:
        idx = np.argmin(np.abs(kmf_all.survival_function_.index - target_year))
        surv_all = kmf_all.survival_function_.iloc[idx, 0]
    obs_all = 1 - surv_all
    pred_mean = np.mean(pred_risk)
    oe = obs_all / pred_mean if pred_mean > 0 else np.nan

    return {
        "slope": reg.coef_[0], "intercept": reg.intercept_,
        "oe_ratio": oe, "observed_overall": obs_all,
        "predicted_overall": pred_mean, "cal_data": cal_df,
    }


def compute_brier(pred_risk, df, target_year):
    """Compute Brier score and Index of Prediction Accuracy (IPA)."""
    event_at_t = ((df["time"] <= target_year) & (df["status"] == 1)).astype(int).values
    brier = np.mean((pred_risk - event_at_t) ** 2)
    rate = event_at_t.mean()
    brier_null = rate * (1 - rate)
    ipa = 1 - brier / brier_null if brier_null > 0 else np.nan
    return brier, ipa, rate


# ========================= Plotting =========================

def draw_calibration(ax, cal, pred_risk, df, year, color, title):
    """Draw a single calibration subplot."""
    if cal is None:
        ax.text(0.5, 0.5, f"{title}\n(Insufficient data)",
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        return

    cd = cal["cal_data"]
    mx = max(cd["predicted"].max(), cd["observed"].max()) * 1.15

    # Perfect calibration line
    ax.plot([0, mx], [0, mx], "--", color="#666", lw=1.5, alpha=0.7,
            label="Perfect calibration", zorder=1)

    # Fitted calibration line
    cv = cd[(cd["observed"] > 0) & (cd["observed"] < 1)]
    if len(cv) >= 2:
        z = np.polyfit(cv["predicted"], cv["observed"], 1)
        xl = np.linspace(0, mx, 100)
        ax.plot(xl, np.polyval(z, xl), "-", color=color, lw=2.5,
                label=f"Fitted (slope = {cal['slope']:.2f})", zorder=2)

    # Error bars
    if not cd["obs_lower"].isna().all():
        yerr_low = np.clip(cd["observed"].values - cd["obs_lower"].fillna(cd["observed"]).values, 0, None)
        yerr_high = np.clip(cd["obs_upper"].fillna(cd["observed"]).values - cd["observed"].values, 0, None)
        ax.errorbar(cd["predicted"], cd["observed"],
                    yerr=[yerr_low, yerr_high],
                    fmt="none", color=color, alpha=0.4, capsize=3, zorder=3)

    # Scatter points (sized by group n)
    sizes = cd["n"] / cd["n"].max() * 180 + 35
    ax.scatter(cd["predicted"], cd["observed"], s=sizes, c=color,
               alpha=0.85, edgecolors="white", lw=1.5, label="Decile groups", zorder=4)

    ax.set_xlim([-mx * 0.02, mx])
    ax.set_ylim([-mx * 0.02, mx])
    ax.set_xlabel(f"Predicted {year}-year risk")
    ax.set_ylabel(f"Observed {year}-year risk")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.set_aspect("equal", adjustable="box")

    # Metrics text box
    ev = int(((df["time"] <= year) & (df["status"] == 1)).sum())
    try:
        c = concordance_index(df["time"], -pred_risk, df["status"])
    except Exception:
        c = np.nan
    brier, ipa, _ = compute_brier(pred_risk, df, year)

    txt = (f"Slope = {cal['slope']:.2f}\n"
           f"O:E = {cal['oe_ratio']:.2f}\n"
           f"C = {c:.2f}\n"
           f"Brier = {brier:.4f}\n"
           f"IPA = {ipa * 100:.1f}%\n"
           f"N = {len(df):,}\n"
           f"Events = {ev:,}")
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=7.5,
            va="top", bbox=dict(boxstyle="round,pad=0.5", fc="white",
                                ec=color, alpha=0.92, lw=1.2),
            family="monospace")
    ax.legend(loc="lower right", framealpha=0.92, fontsize=7)
    ax.grid(True, alpha=0.25, ls=":")


def figure1(all_results, path):
    """Figure 1: 2x3 calibration plots for 6 cohorts."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for idx, (name, r) in enumerate(all_results.items()):
        if idx >= 6:
            break
        cfg = COHORT_CONFIG[name]
        draw_calibration(axes[idx], r["cal"], r["pred"], r["df"],
                         r["year"], cfg["color"],
                         f"{cfg['label']} ({r['year']}-year)")

    for idx in range(len(all_results), 6):
        axes[idx].set_visible(False)

    plt.suptitle("Figure 1. Calibration of BrainVital8 across six cohorts",
                 fontsize=15, fontweight="bold", y=1.00)
    plt.tight_layout()
    plt.savefig(path + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(path + ".pdf", bbox_inches="tight")
    plt.close()
    print(f"  Figure 1 saved: {path}.png/.pdf")


def figure2(all_results, path):
    """Figure 2: Forest plot with 4 panels (Slope, O:E, C-stat, IPA)."""
    fig, axes = plt.subplots(1, 4, figsize=(22, 6), sharey=True)
    names = list(all_results.keys())
    n = len(names)
    y = np.arange(n)

    slopes, oes, cs, ipas = [], [], [], []
    for nm in names:
        r = all_results[nm]
        cr = r["cal"]
        if cr is None:
            slopes.append(np.nan); oes.append(np.nan)
            cs.append(np.nan); ipas.append(np.nan)
        else:
            slopes.append(cr["slope"]); oes.append(cr["oe_ratio"])
            try:
                cs.append(concordance_index(r["df"]["time"], -r["pred"], r["df"]["status"]))
            except Exception:
                cs.append(np.nan)
            _, ipa, _ = compute_brier(r["pred"], r["df"], r["year"])
            ipas.append(ipa * 100)

    colors = [COHORT_CONFIG[nm]["color"] for nm in names]
    labels = [f"{COHORT_CONFIG[nm]['label']}\n({all_results[nm]['year']}-y)" for nm in names]

    # Panel A: Calibration Slope
    ax = axes[0]
    ax.axvline(1, color="gray", ls="--", lw=1.5, alpha=0.7)
    ax.axvspan(0.85, 1.15, alpha=0.12, color="green", label="Acceptable")
    for i, (s, col) in enumerate(zip(slopes, colors)):
        if not np.isnan(s):
            ax.scatter(s, y[i], s=200, color=col, edgecolor="black", lw=1.5, zorder=3)
            ax.text(s + 0.03, y[i], f"{s:.2f}", va="center", fontsize=9, fontweight="bold")
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlabel("Calibration Slope"); ax.set_title("A. Calibration Slope", fontweight="bold")
    ax.legend(loc="lower right", fontsize=8); ax.grid(True, alpha=0.3, axis="x")
    ax.set_xlim([0.3, 1.6])

    # Panel B: O:E Ratio
    ax = axes[1]
    ax.axvline(1, color="gray", ls="--", lw=1.5, alpha=0.7)
    ax.axvspan(0.85, 1.15, alpha=0.12, color="green", label="Acceptable")
    for i, (o, col) in enumerate(zip(oes, colors)):
        if not np.isnan(o):
            ax.scatter(o, y[i], s=200, color=col, edgecolor="black", lw=1.5, zorder=3)
            ax.text(o + 0.03, y[i], f"{o:.2f}", va="center", fontsize=9, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xlabel("Observed-to-Expected Ratio"); ax.set_title("B. O:E Ratio", fontweight="bold")
    ax.legend(loc="lower right", fontsize=8); ax.grid(True, alpha=0.3, axis="x")
    ax.set_xlim([0.5, 1.6])

    # Panel C: C-statistic
    ax = axes[2]
    ax.axvline(0.5, color="gray", ls="--", lw=1.5, alpha=0.5)
    ax.axvspan(0.7, 0.8, alpha=0.12, color="green", label="Good")
    ax.axvspan(0.8, 1.0, alpha=0.12, color="blue", label="Excellent")
    for i, (cv, col) in enumerate(zip(cs, colors)):
        if not np.isnan(cv):
            ax.scatter(cv, y[i], s=200, color=col, edgecolor="black", lw=1.5, zorder=3)
            ax.text(cv + 0.01, y[i], f"{cv:.2f}", va="center", fontsize=9, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xlabel("C-statistic (OOF)"); ax.set_title("C. Discrimination", fontweight="bold")
    ax.legend(loc="lower right", fontsize=8); ax.grid(True, alpha=0.3, axis="x")
    ax.set_xlim([0.5, 0.95])

    # Panel D: IPA
    ax = axes[3]
    ax.axvline(0, color="gray", ls="--", lw=1.5, alpha=0.7, label="Null")
    ax.axvspan(0, 5, alpha=0.08, color="#F4D03F", label="Marginal (0-5%)")
    ax.axvspan(5, 15, alpha=0.12, color="#27AE60", label="Moderate (5-15%)")
    ax.axvspan(15, 100, alpha=0.15, color="#2471A3", label="Good (>15%)")
    for i, (ip, col) in enumerate(zip(ipas, colors)):
        if not np.isnan(ip):
            ax.scatter(ip, y[i], s=200, color=col, edgecolor="black", lw=1.5, zorder=3)
            ax.text(ip + 0.5, y[i], f"{ip:.1f}%", va="center", fontsize=9, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xlabel("IPA (%)"); ax.set_title("D. Overall Performance (IPA)", fontweight="bold")
    ax.legend(loc="lower right", fontsize=7); ax.grid(True, alpha=0.3, axis="x")
    iv = [i for i in ipas if not np.isnan(i)]
    if iv:
        ax.set_xlim([min(-2, min(iv) - 2), max(10, max(iv) + 3)])

    plt.suptitle("Figure 2. Summary of BrainVital8 performance across six cohorts",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(path + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(path + ".pdf", bbox_inches="tight")
    plt.close()
    print(f"  Figure 2 saved: {path}.png/.pdf")


def figure_individual(all_results, outdir):
    """Individual large calibration plots (supplementary material)."""
    for name, r in all_results.items():
        if r["cal"] is None:
            continue
        cfg = COHORT_CONFIG[name]
        fig, ax = plt.subplots(figsize=(7.5, 7.5))
        draw_calibration(ax, r["cal"], r["pred"], r["df"],
                         r["year"], cfg["color"],
                         f"{cfg['label']} ({r['year']}-year)")
        plt.tight_layout()
        p = os.path.join(outdir, f"SupplFig_{cfg['label']}")
        plt.savefig(p + ".png", dpi=300, bbox_inches="tight")
        plt.savefig(p + ".pdf", bbox_inches="tight")
        plt.close()
        print(f"  Supplementary figure saved: {p}.png/.pdf")


# ========================= Main =========================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 80)
    print("  BrainVital8 Calibration Analysis")
    print("=" * 80)

    cohorts = load_and_split(DATA_FILE)
    all_results = {}
    summary_rows = []

    for name in COHORT_CONFIG:
        if name not in cohorts:
            print(f"\n  Warning: {name} not found in data, skipping")
            continue

        cfg = COHORT_CONFIG[name]
        df = cohorts[name]
        year = cfg["time_point"]

        print(f"\nProcessing: {cfg['label']} ({year}-year), N={len(df):,}, "
              f"events={int(df['status'].sum()):,}")

        if year > df["time"].max() * 0.95:
            old = year
            year = int(df["time"].max() * 0.9)
            print(f"  Warning: {old}y exceeds follow-up limit, using {year}y instead")

        pred = get_oof_predictions(df, COVARIATES, year, k=N_FOLDS)
        cal = km_based_calibration(pred, df["time"].values,
                                   df["status"].values, year, n_groups=N_GROUPS)

        all_results[name] = {"df": df, "pred": pred, "cal": cal, "year": year}

        if cal:
            try:
                c = concordance_index(df["time"], -pred, df["status"])
            except Exception:
                c = np.nan
            ev = int(((df["time"] <= year) & (df["status"] == 1)).sum())
            brier, ipa, rate = compute_brier(pred, df, year)
            print(f"  Slope={cal['slope']:.3f}, O:E={cal['oe_ratio']:.3f}, "
                  f"C={c:.3f}, Brier={brier:.5f}, IPA={ipa * 100:.2f}%")
            summary_rows.append({
                "Cohort": cfg["label"], "Time (years)": year,
                "N": len(df), "Events": ev,
                "Event Rate (%)": round(rate * 100, 3),
                "Slope": round(cal["slope"], 3),
                "Intercept": round(cal["intercept"], 5),
                "O:E": round(cal["oe_ratio"], 3),
                "C-statistic": round(c, 3),
                "Brier": round(brier, 5),
                "IPA (%)": round(ipa * 100, 2),
            })

    # Summary table
    sdf = pd.DataFrame(summary_rows)
    sdf.to_csv(os.path.join(OUTPUT_DIR, "Table1_BrainVital8.csv"),
               index=False, encoding="utf-8-sig")

    # Figures
    print("\nGenerating figures...")
    figure1(all_results, os.path.join(OUTPUT_DIR, "Figure1_calibration"))
    figure2(all_results, os.path.join(OUTPUT_DIR, "Figure2_forest"))
    figure_individual(all_results, OUTPUT_DIR)

    # Print summary
    print("\n" + "=" * 100)
    print("  Table 1: BrainVital8 Performance Summary")
    print("=" * 100)
    print(sdf.to_string(index=False))
    print(f"\nAll outputs saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()